In [1]:
import pandas as pd

In [2]:
# --- Configuration ---
FILE_NAME = 'notebooks/data/carolina_beach/flood_events.csv'  # Replace with your actual file name
CENTER_OF_ROADWAY_ELEVATION = 3.277  # Replace with your specific elevation (e.g., in feet or meters matching your data)

def process_flood_data(file_path, roadway_elevation):
    # 1. Load the data
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        return

    # Convert time column to datetime objects to allow for time calculations
    # 'mixed' format helps pandas infer the format automatically
    df['time_UTC'] = pd.to_datetime(df['time_UTC'], format='mixed')

    # Sort by event and time to ensure correct order for duration calculation
    df = df.sort_values(by=['flood_event', 'time_UTC'])

    # 2. Create the binary column (1 if sensor level > roadway elevation, else 0)
    df['is_inundated'] = (df['sensor_water_level'] > roadway_elevation).astype(int)

    # 3. Calculate summary statistics for duration
    # We calculate the time difference between the current row and the previous row.
    # We assume that if a row is 'inundated', the water was high during the interval leading up to it.
    
    summary_list = []

    # Group by flood_event to handle each event separately
    for event_id, group in df.groupby('flood_event'):
        # Calculate the time difference (dt) from the previous timestamp in seconds
        # .shift() allows us to access the previous row's value
        time_diff_seconds = group['time_UTC'].diff().dt.total_seconds().fillna(0)
        
        # We only sum the duration if the current point shows inundation
        # Note: This logic assigns the interval (t_prev, t_curr] to the status at t_curr
        inundated_seconds = time_diff_seconds[group['is_inundated'] == 1].sum()
        
        # Convert to hours for readability
        inundated_hours = inundated_seconds / 3600.0
        
        summary_list.append({
            'flood_event': event_id,
            'inundated_duration_hours': inundated_hours,
            'inundated_duration_minutes': inundated_seconds / 60.0
        })

    # Create a summary DataFrame
    summary_df = pd.DataFrame(summary_list)

    return df, summary_df

# --- Execution ---
if __name__ == "__main__":
    # Process the data
    df_result, summary_result = process_flood_data(FILE_NAME, CENTER_OF_ROADWAY_ELEVATION)

    if df_result is not None:
        # Save the detailed data with the new column
        output_filename = 'flood_data_analyzed.csv'
        df_result.to_csv(output_filename, index=False)
        print(f"Detailed analysis saved to '{output_filename}'")
        
        # Display the Summary Statistics
        print("\n--- Summary Statistics (Duration of Inundation) ---")
        print(summary_result)
        
        # Optional: Print a preview of the inundated rows
        print("\n--- Preview of Inundated Data Points ---")
        print(df_result[df_result['is_inundated'] == 1][['flood_event', 'time_UTC', 'sensor_water_level', 'is_inundated']].head())

Detailed analysis saved to 'flood_data_analyzed.csv'

--- Summary Statistics (Duration of Inundation) ---
    flood_event  inundated_duration_hours  inundated_duration_minutes
0             1                  0.016667                         1.0
1             2                  2.283333                       137.0
2             3                  8.166667                       490.0
3             4                  0.000000                         0.0
4             5                  0.000000                         0.0
5             6                  3.300000                       198.0
6             7                  3.500000                       210.0
7             8                  0.000000                         0.0
8             9                  5.000000                       300.0
9            10                  0.000000                         0.0
10           11                  0.000000                         0.0
11           12                  2.300000             

In [4]:
import pandas as pd

# --- Configuration ---
FILE_NAME = 'notebooks/data/carolina_beach/flood_events.csv'  # Replace with your actual file name
CENTER_OF_ROADWAY_ELEVATION = 3.277  # Replace with your specific elevation (e.g., in feet or meters matching your data)

def process_flood_data(file_path, roadway_elevation):
    # 1. Load the data
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        return

    # Convert time to datetime
    df['time_UTC'] = pd.to_datetime(df['time_UTC'], format='mixed')

    # Sort to ensure correct order
    df = df.sort_values(by=['flood_event', 'time_UTC'])

    # 2. Create the binary column
    df['center_roadway_inundated'] = (df['sensor_water_level'] > roadway_elevation).astype(int)

    # 3. Initialize the new summary column with None (empty)
    df['center_roadway_inundation_duration_(hours)'] = None

    # 4. Calculate duration per event and assign to the last row
    for event_id, group in df.groupby('flood_event'):
        # Calculate time difference from previous row in seconds
        dt_seconds = group['time_UTC'].diff().dt.total_seconds().fillna(0)
        
        # Sum the duration where the roadway was inundated
        # This sums the intervals leading up to an inundated timestamp
        inundated_seconds = dt_seconds[group['center_roadway_inundated'] == 1].sum()
        inundated_hours = inundated_seconds / 3600.0
        
        # Find the index of the last row for this event
        last_index = group.index[-1]
        
        # Assign the calculated value to that specific cell in the main DataFrame
        df.at[last_index, 'center_roadway_inundation_duration_(hours)'] = inundated_hours
        df.at[last_index, 'center_roadway_inundation_duration_(minutes)'] = inundated_seconds / 60.0

    return df

# --- Execution ---
if __name__ == "__main__":
    df_result = process_flood_data(FILE_NAME, CENTER_OF_ROADWAY_ELEVATION)

    if df_result is not None:
        output_filename = 'flood_data_analyzed.csv'
        df_result.to_csv(output_filename, index=False)
        print(f"Analysis complete. Data saved to '{output_filename}'")
        
        # Preview the end of the first event to show the new summary value
        print("\n--- Preview of Last 5 Rows (showing summary column) ---")
        cols_to_show = ['flood_event', 'time_UTC', 'sensor_water_level', 'center_roadway_inundated', 'center_roadway_inundation_duration_(hours)']
        print(df_result[cols_to_show].head(30).tail())

Analysis complete. Data saved to 'flood_data_analyzed.csv'

--- Preview of Last 5 Rows (showing summary column) ---
    flood_event                  time_UTC  sensor_water_level  \
25            1 2022-08-19 17:05:26+00:00            3.043431   
26            1 2022-08-19 17:06:26+00:00            3.060978   
27            1 2022-08-19 17:07:26+00:00            3.113345   
28            1 2022-08-19 17:08:26+00:00            3.015215   
29            2 2022-09-29 15:26:27+00:00            3.080153   

    center_roadway_inundated center_roadway_inundation_duration_(hours)  
25                         0                                       None  
26                         0                                       None  
27                         0                                       None  
28                         0                                   0.016667  
29                         0                                       None  


In [6]:
import pandas as pd

# --- Configuration ---
FILE_NAME = 'notebooks/data/down_east/flood_events.csv'  # Replace with your actual file name
CENTER_OF_ROADWAY_ELEVATION = 1.946  # Replace with your specific elevation (e.g., in feet or meters matching your data)
#3.277 CB_03 1.946 DE_01 Crown Elevations

def process_flood_data_robust(file_path, roadway_elevation):
    # 1. Load the data
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        return

    # Convert time to datetime
    df['time_UTC'] = pd.to_datetime(df['time_UTC'], format='mixed')

    # Sort strictly by event and time
    df = df.sort_values(by=['flood_event', 'time_UTC'])

    # 2. Determine status
    # 1 if Inundated, 0 if not
    df['is_inundated'] = (df['sensor_water_level'] > roadway_elevation).astype(int)

    # 3. Calculate Intervals and Duration
    # Calculate time difference between current row and previous row (in seconds)
    df['dt_seconds'] = df.groupby('flood_event')['time_UTC'].diff().dt.total_seconds().fillna(0)

    # shift(1) gets the "is_inundated" status of the PREVIOUS row
    # We group by flood_event to ensure we don't shift data from Event 1 into Event 2
    df['prev_inundated'] = df.groupby('flood_event')['is_inundated'].shift(1).fillna(0)

    # 4. Calculate contribution of each interval
    # Logic: We only add the duration of the interval (dt) if the water was ALREADY high 
    # at the start of the interval (prev_inundated == 1).
    df['segment_duration_seconds'] = df['dt_seconds'] * df['prev_inundated']

    # 5. Sum up the durations per event and assign to the summary column
    df['center_roadway_inundation_duration_(hours)'] = None

    for event_id, group in df.groupby('flood_event'):
        # Sum the valid segments
        total_seconds = group['segment_duration_seconds'].sum()
        total_hours = total_seconds / 3600.0
        
        # Assign to the last row of the event
        last_index = group.index[-1]
        df.at[last_index, 'center_roadway_inundation_duration_(hours)'] = total_hours
        df.at[last_index, 'center_roadway_inundation_duration_(minutes)'] = total_seconds / 60.0

    # Cleanup helper columns if desired (optional)
    df.drop(columns=['dt_seconds', 'prev_inundated', 'segment_duration_seconds'], inplace=True)

    return df

# --- Execution ---
if __name__ == "__main__":
    df_result = process_flood_data_robust(FILE_NAME, CENTER_OF_ROADWAY_ELEVATION)

    if df_result is not None:
        output_filename = 'flood_data_analyzed_v2.csv'
        df_result.to_csv(output_filename, index=False)
        print(f"Analysis complete. Data saved to '{output_filename}'")
        
        # Preview to verify the fix
        print("\n--- Summary Preview (Last rows of events) ---")
        cols = ['flood_event', 'time_UTC', 'sensor_water_level', 'is_inundated', 'center_roadway_inundation_duration_(hours)']
        # Show rows where the summary is present (not null)
        print(df_result.dropna(subset=['center_roadway_inundation_duration_(hours)'])[cols].head())

Analysis complete. Data saved to 'flood_data_analyzed_v2.csv'

--- Summary Preview (Last rows of events) ---
     flood_event                  time_UTC  sensor_water_level  is_inundated  \
50             1 2024-05-22 03:44:40+00:00            1.592571             0   
63             2 2024-06-04 01:16:26+00:00            1.594886             0   
97             3 2024-06-05 03:10:26+00:00            1.613766             0   
241            4 2024-07-27 20:49:46+00:00            1.617490             0   
309            5 2024-07-28 21:26:58+00:00            1.618072             0   

    center_roadway_inundation_duration_(hours)  
50                                         1.5  
63                                         0.0  
97                                         0.0  
241                                        0.0  
309                                        0.0  


In [4]:
import pandas as pd

# --- Configuration ---
FILE_NAME = '/home/rmccune/Documents/poseidon/notebooks/data/down_east/flood_events.csv'  # Replace with your actual file name
CENTER_OF_ROADWAY_ELEVATION = 1.946  # Replace with your specific elevation

def process_flood_data_complete(file_path, roadway_elevation):
    # 1. Load the data
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
        return None

    # Convert time to datetime
    df['time_UTC'] = pd.to_datetime(df['time_UTC'], format='mixed')

    # Sort strictly by event and time
    df = df.sort_values(by=['flood_event', 'time_UTC'])

    # ---------------------------------------------------------
    # PART A: Per-Row Calculations
    # ---------------------------------------------------------

    # 1. Binary Inundation (1 if Wet, 0 if Dry)
    df['is_inundated'] = (df['sensor_water_level'] > roadway_elevation).astype(int)

    # 2. Roadway Depth
    # Calculate difference, then use clip(lower=0) to turn negatives into 0
    df['roadway_depth'] = (df['sensor_water_level'] - roadway_elevation).clip(lower=0)

    # ---------------------------------------------------------
    # PART B: Duration Logic (Forward-Looking)
    # ---------------------------------------------------------
    
    # Calculate time difference (in seconds)
    df['dt_seconds'] = df.groupby('flood_event')['time_UTC'].diff().dt.total_seconds().fillna(0)

    # Shift 'is_inundated' to see if the PREVIOUS point was wet
    df['prev_inundated'] = df.groupby('flood_event')['is_inundated'].shift(1).fillna(0)

    # Only count time if the PREVIOUS point was already inundated
    df['segment_duration_seconds'] = df['dt_seconds'] * df['prev_inundated']

    # ---------------------------------------------------------
    # PART C: Event Summaries (assigned to last row)
    # ---------------------------------------------------------
    
    # Initialize summary columns with None
    df['center_roadway_inundation_duration_(hours)'] = None
    df['max_center_roadway_depth'] = None

    for event_id, group in df.groupby('flood_event'):
        # 1. Calculate Total Duration
        total_seconds = group['segment_duration_seconds'].sum()
        total_hours = total_seconds / 3600.0
        
        # 2. Calculate Max Depth for this event
        max_event_depth = group['roadway_depth'].max()
        
        # 3. Assign to the last row of the event
        last_index = group.index[-1]
        df.at[last_index, 'center_roadway_inundation_duration_(hours)'] = total_hours
        df.at[last_index, 'max_center_roadway_depth'] = max_event_depth

    # Cleanup helper columns (optional, keeps the file cleaner)
    df.drop(columns=['dt_seconds', 'prev_inundated', 'segment_duration_seconds'], inplace=True)

    return df

# --- Execution ---
if __name__ == "__main__":
    df_result = process_flood_data_complete(FILE_NAME, CENTER_OF_ROADWAY_ELEVATION)

    if df_result is not None:
        output_filename = 'flood_data_analyzed_complete.csv'
        df_result.to_csv(output_filename, index=False)
        print(f"Analysis complete. Data saved to '{output_filename}'")
        
        # Preview the specific columns we just added
        print("\n--- Summary Preview (Last rows of events) ---")
        cols = [
            'flood_event', 
            'time_UTC', 
            'sensor_water_level', 
            'roadway_depth', 
            'max_center_roadway_depth', 
            'center_roadway_inundation_duration_(hours)'
        ]
        # Show rows where the summary is present
        print(df_result.dropna(subset=['max_center_roadway_depth'])[cols].head())

Analysis complete. Data saved to 'flood_data_analyzed_complete.csv'

--- Summary Preview (Last rows of events) ---
     flood_event                  time_UTC  sensor_water_level  roadway_depth  \
50             1 2024-05-22 03:44:40+00:00            1.592571            0.0   
63             2 2024-06-04 01:16:26+00:00            1.594886            0.0   
97             3 2024-06-05 03:10:26+00:00            1.613766            0.0   
241            4 2024-07-27 20:49:46+00:00            1.617490            0.0   
309            5 2024-07-28 21:26:58+00:00            1.618072            0.0   

    max_center_roadway_depth center_roadway_inundation_duration_(hours)  
50                  0.083961                                        1.5  
63                       0.0                                        0.0  
97                       0.0                                        0.0  
241                      0.0                                        0.0  
309                      0.0